In [1]:
import os
import random
import numpy as np
import pandas as pd
import cv2
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, cohen_kappa_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# File Paths
DATA_ROOT = Path('../data')  # <-- Set to your local data folder
CSV_PATH = DATA_ROOT / 'label' / 'trainLabels.csv'
IMG_DIR = Path('../data/processed/clahe')  # <-- Set to your local CLAHE images folder

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4

# Detect image extension from the CLAHE folder
_sample = next(IMG_DIR.iterdir())
IMG_EXT = _sample.suffix
print(f'Image extension: {IMG_EXT}')

# Load Data
df = pd.read_csv(CSV_PATH)
df['image_path'] = df['image'].astype(str).apply(lambda x: str(IMG_DIR / f'{x}{IMG_EXT}'))
df['label'] = df['level'].astype(int)
df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)
print(f'Total images found: {len(df)}')

# 1. Main Splits
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)

# 2. Stage 1: Binary (0 = Healthy, 1 = Diseased)
for d in [train_df, val_df, test_df]:
    d['label_s1'] = (d['label'] > 0).astype(int)

# 3. Stage 2: Diseased only (classes 1-4 mapped to 0-3)
train_df_s2 = train_df[train_df['label'] > 0].copy()
train_df_s2['label_s2'] = train_df_s2['label'] - 1

val_df_s2 = val_df[val_df['label'] > 0].copy()
val_df_s2['label_s2'] = val_df_s2['label'] - 1

# Weighted sampler for Stage 2 class imbalance
class_counts = train_df_s2['label_s2'].value_counts().sort_index().values
class_weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = [class_weights[label] for label in train_df_s2['label_s2']]
sampler_s2 = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Stage 2 train (diseased only): {len(train_df_s2)}')

/tmp/ipykernel_90966/4224996259.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Device: cuda
Image extension: .jpeg
Total images found: 35126
Train: 24588 | Val: 5269 | Test: 5269
Stage 2 train (diseased only): 6521


In [2]:
class FundusDataset(Dataset):
    """Loads CLAHE-processed images and normalizes on-the-fly."""
    def __init__(self, df, label_col, augment=False):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col
        self.augment = augment

        # --- UPGRADED AUGMENTATIONS ---
        self.train_tf = transforms.Compose([
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), 
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2), 
        ])

        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image_path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        if self.augment:
            x = self.train_tf(x)
        x = self.normalize(x)

        y = torch.tensor(int(row[self.label_col]), dtype=torch.long)
        return x, y

# Stage 1 Loaders
train_loader_s1 = DataLoader(
    FundusDataset(train_df, 'label_s1', augment=True),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader_s1 = DataLoader(
    FundusDataset(val_df, 'label_s1', augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

# Stage 2 Loaders
train_loader_s2 = DataLoader(
    FundusDataset(train_df_s2, 'label_s2', augment=True),
    batch_size=BATCH_SIZE,
    sampler=sampler_s2,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader_s2 = DataLoader(
    FundusDataset(val_df_s2, 'label_s2', augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print('DataLoaders ready.')

DataLoaders ready.


In [3]:
import torch.nn.functional as F

# --- NEW: Custom Focal Loss to handle hard examples ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss
        
class Stage1Gatekeeper(nn.Module):
    # Binary Classifier (Healthy vs Diseased)
    def __init__(self):
        super().__init__()
        # --- UPGRADED TO B3 ---
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 1)
        )

    def forward(self, x):
        return self.backbone(x).squeeze(1)

class Stage2Expert(nn.Module):
    # 4-Class Classifier (Mild, Mod, Severe, Proliferative)
    def __init__(self):
        super().__init__()
        # --- UPGRADED TO B3 ---
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),  
            nn.Linear(in_features, 4)
        )

    def forward(self, x):
        return self.backbone(x)

In [4]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs,
                is_binary=False, patience=5):
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    best_loss = float('inf')
    best_weights = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if is_binary: y = y.float()

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)
        scheduler.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                if is_binary: y = y.float()
                logits = model(x)
                loss = criterion(logits, y)
                val_loss += loss.item() * x.size(0)
        val_loss /= len(val_loader.dataset)

        lr_now = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{epochs} - Train: {train_loss:.4f} - Val: {val_loss:.4f} - LR: {lr_now:.6f}")

        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    model.load_state_dict(best_weights)
    return model

In [5]:
# ==========================================
# TRAIN STAGE 1: THE GATEKEEPER
# ==========================================
print("--- Training Stage 1: Gatekeeper ---")
gatekeeper = Stage1Gatekeeper().to(DEVICE)

num_healthy = (train_df['label_s1'] == 0).sum()
num_diseased = (train_df['label_s1'] == 1).sum()
pos_weight = torch.tensor([num_healthy / num_diseased]).to(DEVICE)

criterion_s1 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_s1 = torch.optim.AdamW(gatekeeper.parameters(), lr=1e-4)

gatekeeper = train_model(gatekeeper, train_loader_s1, val_loader_s1,
                         criterion_s1, optimizer_s1, epochs=25, is_binary=True, patience=5)


# ==========================================
# TRAIN STAGE 2: THE EXPERT
# ==========================================
print("\n--- Training Stage 2: Expert Grader ---")
expert = Stage2Expert().to(DEVICE)

# --- NEW: Swap CrossEntropy for Focal Loss ---
criterion_s2 = FocalLoss(gamma=2.0) 

optimizer_s2 = torch.optim.AdamW(expert.parameters(), lr=1e-4, weight_decay=1e-2)

expert = train_model(expert, train_loader_s2, val_loader_s2,
                     criterion_s2, optimizer_s2, epochs=30, is_binary=False, patience=5)

--- Training Stage 1: Gatekeeper ---


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /home/cap/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100.0%


Epoch 1/25 - Train: 0.8858 - Val: 0.8084 - LR: 0.000100
Epoch 2/25 - Train: 0.8162 - Val: 0.7935 - LR: 0.000098
Epoch 3/25 - Train: 0.7766 - Val: 0.7847 - LR: 0.000096
Epoch 4/25 - Train: 0.7448 - Val: 0.7770 - LR: 0.000094
Epoch 5/25 - Train: 0.7121 - Val: 0.7587 - LR: 0.000090
Epoch 6/25 - Train: 0.6898 - Val: 0.7618 - LR: 0.000086
Epoch 7/25 - Train: 0.6596 - Val: 0.7474 - LR: 0.000082
Epoch 8/25 - Train: 0.6401 - Val: 0.7751 - LR: 0.000077
Epoch 9/25 - Train: 0.6142 - Val: 0.7718 - LR: 0.000071
Epoch 10/25 - Train: 0.5800 - Val: 0.8074 - LR: 0.000065
Epoch 11/25 - Train: 0.5519 - Val: 0.9154 - LR: 0.000059
Epoch 12/25 - Train: 0.5122 - Val: 0.8733 - LR: 0.000053
  Early stopping at epoch 12 (no improvement for 5 epochs)

--- Training Stage 2: Expert Grader ---
Epoch 1/30 - Train: 0.5900 - Val: 0.4648 - LR: 0.000100
Epoch 2/30 - Train: 0.4423 - Val: 0.4604 - LR: 0.000099
Epoch 3/30 - Train: 0.3939 - Val: 0.4283 - LR: 0.000098
Epoch 4/30 - Train: 0.3393 - Val: 0.3842 - LR: 0.000096
E

In [6]:
from sklearn.metrics import f1_score

print("\n--- Tuning Stage 1 Decision Threshold ---")
gatekeeper.eval()
expert.eval()

val_probs = []
val_targets = []

# Gather all Stage 1 probabilities on the validation set
with torch.no_grad():
    for x, target in val_loader_s1:
        x = x.to(DEVICE)
        logits = gatekeeper(x)
        probs = torch.sigmoid(logits)
        val_probs.extend(probs.cpu().numpy())
        val_targets.extend(target.numpy())

# Sweep thresholds to find the best F1 Score
best_thresh = 0.5
best_f1 = 0
for t in np.arange(0.1, 0.9, 0.05):
    preds = (np.array(val_probs) >= t).astype(int)
    f1 = f1_score(val_targets, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"Optimal Stage 1 Threshold: {best_thresh:.2f} (Val F1: {best_f1:.4f})")

print("\n--- Evaluating the Full Cascade on Test Data ---")
test_loader_final = DataLoader(
    FundusDataset(test_df, 'label', augment=False),
    batch_size=1, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

y_true = []
y_pred = []

with torch.no_grad():
    for x, target in test_loader_final:
        x = x.to(DEVICE)

        # 1. Ask the Gatekeeper
        s1_logit = gatekeeper(x)
        s1_prob = torch.sigmoid(s1_logit)

        # --- NEW: Use the dynamic threshold here ---
        if s1_prob.item() < best_thresh:
            final_pred = 0
        else:
            # 2. Send to Expert
            s2_logits = expert(x)
            s2_pred = torch.argmax(s2_logits, dim=1).item()
            final_pred = s2_pred + 1

        y_true.append(target.item())
        y_pred.append(final_pred)

print(f'\nFinal Cascade Accuracy: {accuracy_score(y_true, y_pred):.4f}')
print(f'Final Cascade QWK: {cohen_kappa_score(y_true, y_pred, weights="quadratic"):.4f}')
print('\nFinal Confusion Matrix:\n', confusion_matrix(y_true, y_pred))
print('\nFinal Classification Report:\n', classification_report(y_true, y_pred))


--- Tuning Stage 1 Decision Threshold ---
Optimal Stage 1 Threshold: 0.60 (Val F1: 0.6372)

--- Evaluating the Full Cascade on Test Data ---

Final Cascade Accuracy: 0.7239
Final Cascade QWK: 0.6404

Final Confusion Matrix:
 [[3358  316  153   28   17]
 [ 263   66   29    6    2]
 [ 259  140  258   94   43]
 [   8    5   33   76    9]
 [  11    0   21   18   56]]

Final Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.87      0.86      3872
           1       0.13      0.18      0.15       366
           2       0.52      0.32      0.40       794
           3       0.34      0.58      0.43       131
           4       0.44      0.53      0.48       106

    accuracy                           0.72      5269
   macro avg       0.46      0.50      0.46      5269
weighted avg       0.74      0.72      0.73      5269

